# AISlopDetector Training on Kaggle GPU
Free GPU training (T4 x2 weekly, or P100). CIFAKE dataset is pre-loaded from Kaggle.

In [ ]:
# Install dependencies
!pip install -q timm mlflow torchvision

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import timm
import numpy as np
import random

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
# CIFAKE is a Kaggle dataset — add it in the notebook sidebar
# Go to Add Data → search "cifake" → add birdy654/cifake-real-and-ai-generated-synthetic-images
# Path will be: /kaggle/input/cifake-real-and-ai-generated-synthetic-images/

import os
DATA_DIR = "/kaggle/input/cifake-real-and-ai-generated-synthetic-images"

if not os.path.exists(DATA_DIR):
    print("Add the CIFAKE dataset using the 'Add Data' button in the right sidebar")
    print("Search for: cifake-real-and-ai-generated-synthetic-images")
else:
    for split in ["train", "test"]:
        for cls in ["REAL", "FAKE"]:
            d = os.path.join(DATA_DIR, split, cls)
            if os.path.isdir(d):
                print(f"{split}/{cls}: {len(os.listdir(d))} images")
    print("Dataset ready!")

In [ ]:
# Config
IMAGE_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 1e-4
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# Dataset
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE + 32),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class CIFAKE(torch.utils.data.Dataset):
    def __init__(self, root, split="train", transform=None):
        self.images = []
        self.labels = []
        self.transform = transform
        for cls, label in [("REAL", 0), ("FAKE", 1)]:
            cls_dir = Path(root) / split / cls
            if cls_dir.is_dir():
                for img_path in cls_dir.glob("*.jpg"):
                    self.images.append(str(img_path))
                    self.labels.append(label)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

full_dataset = CIFAKE(DATA_DIR, split="train", transform=train_transform)
val_size = int(len(full_dataset) * 0.1)
train_size = len(full_dataset) - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])
val_ds.dataset.transform = val_transform

test_ds = CIFAKE(DATA_DIR, split="test", transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {train_size}, Val: {val_size}, Test: {len(test_ds)}")

In [ ]:
# Model
class AISlopClassifier(nn.Module):
    def __init__(self, backbone="efficientnet_b3", num_classes=2):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=True)
        in_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes),
        )

    def forward(self, x):
        return self.backbone(x)

model = AISlopClassifier().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_acc = 0.0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = train_correct = train_total = 0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch} train", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    model.eval()
    val_loss = val_correct = val_total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch} val", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    scheduler.step()
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total

    history.append({"epoch": epoch, "train_acc": train_acc, "val_acc": val_acc})
    print(f"Epoch {epoch}: train_acc={train_acc:.4f}  val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "val_accuracy": val_acc}, "best_model.pth")
        print(f"  -> Saved best model (acc={val_acc:.4f})")

print(f"\nTraining complete. Best val acc: {best_val_acc:.4f}")

In [ ]:
# Evaluation on test set
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

model.load_state_dict(torch.load("best_model.pth", weights_only=True)["model_state_dict"])
model.eval()

all_preds, all_probs, all_labels = [], [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating"):
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

print(f"Test Accuracy:  {accuracy_score(all_labels, all_preds):.4f}")
print(f"Test F1:        {f1_score(all_labels, all_preds):.4f}")
print(f"Test ROC-AUC:   {roc_auc_score(all_labels, [p[1] for p in all_probs]):.4f}")
print(f"\nConfusion Matrix (REAL=0, FAKE=1):")
print(confusion_matrix(all_labels, all_preds))

In [ ]:
# Download the trained model
from IPython.display import FileLink
FileLink("best_model.pth")